# Review-quality model: fine-tune a transformer (Google Colab)

Trains multilingual transformers to predict the review-quality score (1 to 10) from the review text,
on the dataset scored by Gemini, and tests them on the 400 main-run reviews.

**Before running:** menu **Runtime > Change runtime type > T4 GPU > Save**. Then run the cells from top to bottom
(**Runtime > Run all**). Each model takes about 10 minutes.

In [ ]:
# 1. Check the GPU. This should show a Tesla T4 (or another GPU). If it fails, set the runtime type to GPU.
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Get the project (code + scored dataset) from GitHub
!rm -rf ai-review-trends
!git clone -q -b Mastura --depth 1 https://github.com/Masturajannat/ai-review-trends.git
%cd ai-review-trends/quality_model
!pip -q install sentencepiece
!ls scripts

## Train the models
Each cell trains one model and prints its results next to the baseline and the Gemini yardstick.

In [ ]:
# 3a. XLM-RoBERTa base: the standard multilingual model
!python scripts/03_train_transformer.py --model xlm-roberta-base

In [ ]:
# 3b. MuRIL: trained on Indian languages including Bangla and their romanised forms (Banglish)
!python scripts/03_train_transformer.py --model google/muril-base-cased

In [ ]:
# 3c. BanglaBERT (BUET): Bangla-only pre-training
!python scripts/03_train_transformer.py --model csebuetnlp/banglabert

## Results

In [ ]:
# 4. All models side by side (test set = the 400 main-run reviews)
import pandas as pd
t = pd.read_csv("results/model_comparison.csv")
t[["model", "MAE", "RMSE", "within 1", "within 0.5", "Spearman", "best_epoch"]].sort_values("MAE")

In [ ]:
# 5. Download the results (small files) to your computer.
#    Unzip into ai-review-trends/quality_model/results on your laptop, then commit and push.
!zip -q -r transformer_results.zip results -i "results/transformer_*" "results/model_comparison.csv"
from google.colab import files
files.download("transformer_results.zip")

## Keep the best model (optional, needed later for the app)
The fine-tuned models are in `models/` (about 1 GB each) and disappear when the Colab session ends.
The cell below copies **one** of them to your Google Drive. Change `BEST` to the model with the lowest MAE above.

In [ ]:
BEST = "xlm-roberta-base"          # or "google_muril-base-cased" or "csebuetnlp_banglabert"
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p "/content/drive/MyDrive/review_quality_model"
!cp -r "models/{BEST}" "/content/drive/MyDrive/review_quality_model/"
!ls -lh "/content/drive/MyDrive/review_quality_model/{BEST}"